In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, davies_bouldin_score

datasets = {
    "full": "../../data/processed/dataset_full_preprocessed.csv",
    "pca": "../../data/reduced/PCA_data.csv",
    "fs": "../../data/reduced/dataset_fs.csv"
}

outDir = "../../results/kmeans"

In [26]:
ks = range(2,8)

def run_kmeans(dataset_name, df):
    silhouette_scores = []

    for k in ks:
        kmeans = KMeans(
            n_clusters=k,
            random_state=42,
            n_init=20
        )
        labels = kmeans.fit_predict(df)
        score = silhouette_score(df, labels)
        silhouette_scores.append(score)

    best_k = ks[np.argmax(silhouette_scores)]
    best_silhouette = max(silhouette_scores)
    
    kmeans_final = KMeans(
        n_clusters=6,
        random_state=42,
        n_init=20
    )
    final_labels = kmeans_final.fit_predict(df)
    
    db_index = davies_bouldin_score(df, final_labels)

    result_df = df.copy()
    result_df["cluster"] = final_labels

    result_path = f"{outDir}/kmeans_{dataset_name}_clusters.csv"
    result_df.to_csv(result_path, index=False)

    summary = {
        "dataset": dataset_name,
        "best_k": best_k,
        "silhouette": best_silhouette,
        "davies_bouldin": db_index
    }

    return summary

In [27]:
all_results = []

for name, path in datasets.items():
    df = pd.read_csv(path)
    if "Class" in df.columns:
        df = df.drop(columns=["Class"])
    summary = run_kmeans(name, df)
    all_results.append(summary)

In [28]:
summary_df = pd.DataFrame(all_results)
summary_df.to_csv(f"{outDir}/kmeans_summary.csv", index=False)

print(summary_df)

  dataset  best_k  silhouette  davies_bouldin
0    full       2    0.999149        0.677443
1     pca       2    0.999864        0.441329
2      fs       2    0.999526        0.471490
